# Notebook 11 – Handling Imbalanced Data
**File:** `11_Imbalanced_Data.ipynb`

## Learning objectives
You will learn what class imbalance is, why accuracy can be misleading, and how to compare undersampling, oversampling, SMOTE, Borderline-SMOTE, and class weights.

## 1. What is Class Imbalance?
A classification dataset is imbalanced when one class has many more observations than another.

Example: In a fraud-detection dataset, 99% of transactions may be genuine and only 1% may be fraudulent. The genuine class is the majority class; fraud is the minority class.

## 2. Balanced vs Imbalanced Dataset

| Dataset | Class 0 | Class 1 |
|---|---:|---:|
| Balanced example | 500 (50%) | 500 (50%) |
| Imbalanced example | 990 (99%) | 10 (1%) |

A dataset does not need to be exactly 50:50 to be useful. Whether imbalance is a problem depends on the task, error costs, and evaluation goal.

## 3. Why is Class Imbalance a Problem?
A model trained on imbalanced data may focus on the majority class and perform poorly on the minority class. This is especially serious when the minority class represents an important event, such as fraud, a defective product, or a rare disease.

### Why accuracy alone can be misleading
Imagine 1,000 transactions: 990 are genuine and 10 are fraudulent. A model that predicts **every transaction as genuine** gets 990 correct predictions.

Accuracy = 990 / 1,000 = 99%.

However, it detects **zero** fraudulent transactions. Its fraud recall is 0%. Therefore, accuracy alone hides the failure to identify the minority class.

Use a confusion matrix, precision, recall, F1-score, and often precision-recall AUC. Choose metrics based on the real-world cost of false positives and false negatives.

## 4. Create an imbalanced sample dataset
We create a synthetic binary classification dataset with approximately 95% majority class and 5% minority class.

We split the data before applying any resampling. Resampling must happen **only on the training data**. Never resample the test set: the test set should represent the original class distribution.

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=3000,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    n_repeated=0,
    n_classes=2,
    weights=[0.95, 0.05],
    class_sep=1.0,
    random_state=42
)

X = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(1, 13)])
y = pd.Series(y, name="target")

print("Full dataset class counts:")
print(y.value_counts())
print("\nFull dataset class percentages:")
print((y.value_counts(normalize=True) * 100).round(2))

Full dataset class counts:
target
0    2838
1     162
Name: count, dtype: int64

Full dataset class percentages:
target
0    94.6
1     5.4
Name: proportion, dtype: float64


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training class counts:")
print(y_train.value_counts())
print("\nTest class counts (keep original distribution):")
print(y_test.value_counts())

Training class counts:
target
0    2128
1     122
Name: count, dtype: int64

Test class counts (keep original distribution):
target
0    710
1     40
Name: count, dtype: int64


## 5. Establish a baseline model
First train a model without resampling. We use a Decision Tree for a simple demonstration.

We will evaluate the minority class using class 1 as the positive class. In a real project, confirm which label represents the important event.

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

def evaluate_model(model, X_eval, y_eval, model_name="Model"):
    predictions = model.predict(X_eval)
    print(f"--- {model_name} ---")
    print("Accuracy :", round(accuracy_score(y_eval, predictions), 4))
    print("Precision:", round(precision_score(y_eval, predictions, zero_division=0), 4))
    print("Recall   :", round(recall_score(y_eval, predictions, zero_division=0), 4))
    print("F1-score :", round(f1_score(y_eval, predictions, zero_division=0), 4))
    print("\nConfusion matrix:")
    print(confusion_matrix(y_eval, predictions))
    print("\nClassification report:")
    print(classification_report(y_eval, predictions, zero_division=0))

baseline_model = DecisionTreeClassifier(random_state=42)
baseline_model.fit(X_train, y_train)
evaluate_model(baseline_model, X_test, y_test, "Baseline Decision Tree")

--- Baseline Decision Tree ---
Accuracy : 0.9493
Precision: 0.5227
Recall   : 0.575
F1-score : 0.5476

Confusion matrix:
[[689  21]
 [ 17  23]]

Classification report:
              precision    recall  f1-score   support

           0       0.98      0.97      0.97       710
           1       0.52      0.57      0.55        40

    accuracy                           0.95       750
   macro avg       0.75      0.77      0.76       750
weighted avg       0.95      0.95      0.95       750



## 6. Undersampling
Undersampling reduces the number of majority-class observations.

**Advantages:** Faster training and a more balanced training set.

**Disadvantages:** Useful majority-class examples may be discarded, potentially reducing the information available to the model.

### Random Undersampling
Randomly selects a smaller sample from the majority class. The test set remains unchanged.

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

under_sampler = RandomUnderSampler(random_state=42)
X_under, y_under = under_sampler.fit_resample(X_train, y_train)

print("Before undersampling:")
print(y_train.value_counts())
print("\nAfter undersampling:")
print(y_under.value_counts())

under_model = DecisionTreeClassifier(random_state=42)
under_model.fit(X_under, y_under)
evaluate_model(under_model, X_test, y_test, "Random Undersampling")

## 7. Oversampling
Oversampling increases the number of minority-class examples.

**Advantages:** Keeps the majority-class observations and gives the model more minority-class training examples.

**Disadvantages:** It can increase training time. Simple duplication may cause overfitting because repeated examples do not add new information.

### Random Oversampling
Randomly duplicates minority-class observations until the requested class balance is reached.

In [ ]:
from imblearn.over_sampling import RandomOverSampler

over_sampler = RandomOverSampler(random_state=42)
X_over, y_over = over_sampler.fit_resample(X_train, y_train)

print("Before oversampling:")
print(y_train.value_counts())
print("\nAfter oversampling:")
print(y_over.value_counts())

over_model = DecisionTreeClassifier(random_state=42)
over_model.fit(X_over, y_over)
evaluate_model(over_model, X_test, y_test, "Random Oversampling")

## 8. SMOTE (Synthetic Minority Over-sampling Technique)
SMOTE creates synthetic minority-class examples. It selects a minority observation and uses nearby minority observations to generate new points between them.

SMOTE is not simply copying rows. It can help when the minority class has enough meaningful neighbours.

**Caution:** SMOTE can create unrealistic examples when features are categorical, when classes overlap, or when data contains outliers. For mixed numeric and categorical data, consider methods such as SMOTENC rather than applying ordinary SMOTE blindly.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, k_neighbors=5)
X_smote, y_smote = smote.fit_resample(X_train, y_train)

print("After SMOTE:")
print(y_smote.value_counts())

smote_model = DecisionTreeClassifier(random_state=42)
smote_model.fit(X_smote, y_smote)
evaluate_model(smote_model, X_test, y_test, "SMOTE")

## 9. Borderline-SMOTE
Borderline-SMOTE focuses synthetic sample generation on minority observations near the class boundary, where classification may be difficult.

It can be useful when many minority observations lie close to the majority class. It is not guaranteed to improve results and may generate noisy or overlapping examples if the class boundary is ambiguous.

In [ ]:
from imblearn.over_sampling import BorderlineSMOTE

borderline_smote = BorderlineSMOTE(
    random_state=42,
    kind="borderline-1"
)
X_border, y_border = borderline_smote.fit_resample(X_train, y_train)

print("After Borderline-SMOTE:")
print(y_border.value_counts())

border_model = DecisionTreeClassifier(random_state=42)
border_model.fit(X_border, y_border)
evaluate_model(border_model, X_test, y_test, "Borderline-SMOTE")

## 10. Class Weights
Class weights tell the learning algorithm to penalize mistakes on some classes more heavily. This can make the model pay more attention to the minority class without changing the number of training rows.

In scikit-learn, `class_weight="balanced"` automatically assigns weights inversely related to class frequencies for estimators that support it.

Class weights are often a good first option because they avoid creating synthetic rows or discarding majority rows. They are not supported by every algorithm, and they still need validation.

In [ ]:
weighted_model = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)
weighted_model.fit(X_train, y_train)
evaluate_model(weighted_model, X_test, y_test, "Decision Tree with class weights")

## 11. Compare the approaches
The following table summarizes the methods. Actual performance depends on the dataset and model; there is no universally best technique.

| Technique | What it does | Main benefit | Main risk |
|---|---|---|---|
| Random Undersampling | Removes majority rows | Faster, balanced training data | Loses information |
| Random Oversampling | Duplicates minority rows | Retains majority data | Can overfit duplicates |
| SMOTE | Creates synthetic minority rows | Adds varied minority examples | Can create unrealistic/overlapping samples |
| Borderline-SMOTE | Focuses on minority boundary examples | Targets difficult regions | Can amplify noisy boundaries |
| Class weights | Increases penalty for minority errors | No resampling required | May need weight/model tuning |

## 12. When should you use each technique?
- **Try class weights first** when your estimator supports them and you want to keep the original training rows.
- **Try random oversampling** when the dataset is small and you want a simple baseline.
- **Try random undersampling** when the majority class is very large and training cost is high.
- **Try SMOTE** for numeric data when the minority class has enough suitable neighbours.
- **Try Borderline-SMOTE** when boundary cases are especially important and the data supports meaningful neighbourhoods.
- **Compare methods with cross-validation** and a validation metric that reflects the business objective.

## 13. Important practical rules
1. Split into training and test sets before resampling.
2. Apply resampling only to training data.
3. For cross-validation, put the sampler inside an `imblearn` pipeline so each fold resamples only its training portion.
4. Keep the test set at its natural class distribution.
5. Do not optimize accuracy alone. Consider recall, precision, F1, PR-AUC, and the costs of each error.
6. Tune the classification threshold using validation data when appropriate.
7. Check whether resampling makes sense for the domain and feature types.

.